# Phase 2 - Support-Conditioned Prototype Segmentation (LG-PM, few-shot track)

**Purpose.** The core few-shot model of the paper. A frozen Phase 1 encoder
provides multi-scale features; each episode builds **foreground/background
masked-average prototypes** from K support image-mask pairs and segments
queries by similarity, optionally fused by a lightweight decoder. Evaluated
with **nested support budgets K in {1,5,10,20}** under three regimes:

1. **In-domain:** supports from TN3K train, queries from TN3K test.
2. **Zero-shot cross-domain:** supports from TN3K train, queries from
   TNUI-2021 / DDTI (no target-domain annotation at all).
3. **K-shot cross-domain adaptation:** supports from the target domain,
   queries held out (TNUI train donors -> TNUI test queries; DDTI-internal
   disjoint support/query sampling).

All K values within an episode share the **same queries and nested
supports** (S1 subset S5 subset S10 subset S20), so K comparisons are
paired by construction - the unit for CIs is the episode.

**Benchmark rows** (Phase 1, frozen):
`SUPERVISED_REF` = supervised ResNet34-UNet trained on all TN3K train masks
(TTA inference): TN3K test 0.8466, TNUI test 0.7464, DDTI 0.7396.
Phase 2 must be read against these - full-data supervision is the upper
reference, not a matched baseline.

**Inputs.** Same mounted datasets; **plus the Phase 1 checkpoint**
`best_resnet34_seed42.pt`: upload it to Kaggle as a dataset (e.g.
`lgpm-ckpts`) and attach it, or keep it in
`/kaggle/working/phase1_segmentation/` if running in the same session.

**Compliance.** The decoder is trained only on TN3K train episodes. Target
domains are used only as evaluation queries and (in the K-shot condition) as
support donors. Fixed seed, fixed episode manifests saved to disk.

## 0. Configuration and checkpoint discovery

In [ ]:
from pathlib import Path
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

INPUT_ROOT = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
PATHS = {
    "tn3k": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "ddti": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "DDTI dataset" / "DDTI",
    "tnui": AUTHOR_ROOT / "tnui-2021" / "TNUI-2021-" / "thyroid_data",
}
OUTPUT_ROOT = Path("/kaggle/working/phase2_prototype_seg")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------- protocol (fixed before any evaluation) --------
IMG_SIZE      = 384
MASK_THRESH   = 127
K_LIST        = [1, 5, 10, 20]        # nested budgets
K_MAX         = max(K_LIST)
Q_QUERY       = 8                     # queries per episode
N_EP_TRAIN    = 300                   # episodes per "epoch"
N_EP_VAL      = 100
N_EP_EVAL     = 200                   # evaluation episodes per condition
EPOCHS        = 30
PATIENCE      = 8
LR_DEC        = 1e-3                  # decoder learning rate
PRED_THRESH   = 0.5                   # fixed; decoder is epistemically a score combiner
USE_DEC_AUG   = True                  # joint light augmentation in training episodes
PROTO_SCALES  = ["e2", "e3", "e4"]    # multi-scale prototype levels (/8, /16, /32)
SCORE_HW      = 48                    # score-map resolution into the decoder

SUPERVISED_REF = {  # Phase 1, TTA, frozen reference rows
    "TN3K test (in-domain)": 0.8466,
    "TNUI-2021 test (ext)": 0.7464,
    "DDTI (ext)": 0.7396,
}

ck_hits = list(Path("/kaggle/input").rglob("best_resnet34_seed42.pt")) + \
          list(Path("/kaggle/working").rglob("best_resnet34_seed42.pt"))
assert ck_hits, (
    "Phase 1 checkpoint not found. Upload best_resnet34_seed42.pt "
    "(from /kaggle/working/phase1_segmentation/) as a Kaggle dataset and attach it, "
    "then re-run this cell.")
CKPT_PATH = ck_hits[0]
print("device:", DEVICE, "| checkpoint:", CKPT_PATH)

## 1. Manifests (identical rules to Phase 1) and episode pools

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def pair_images_masks(image_dir, mask_dir, dataset, split):
    imgs = {p.stem: p for p in image_files(image_dir)}
    msks = {p.stem: p for p in image_files(mask_dir)}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"dataset": dataset, "split": split, "id": k,
                          "image": str(imgs[k]), "mask": str(msks[k])} for k in common])

def add_hashes(df):
    df = df.copy()
    df["sha256"] = df["image"].apply(lambda p: sha256_file(p) if Path(p).exists() else None)
    return df

def drop_exact_duplicates(df, priority):
    df = df.copy()
    df["_p"] = df["split"].map(priority).fillna(99)
    df = df.sort_values(["_p", "id"], kind="stable")
    keep = ~df.duplicated("sha256", keep="first")
    return (df[keep].drop(columns="_p").reset_index(drop=True),
            df[~keep].drop(columns="_p").reset_index(drop=True))

tn3k = pd.concat([
    pair_images_masks(PATHS["tn3k"]/"trainval-image", PATHS["tn3k"]/"trainval-mask", "TN3K", "trainval"),
    pair_images_masks(PATHS["tn3k"]/"test-image",     PATHS["tn3k"]/"test-mask",     "TN3K", "test"),
], ignore_index=True)
tn3k = add_hashes(tn3k)
tn3k, _ = drop_exact_duplicates(tn3k, {"trainval": 0, "test": 1})
rng = np.random.RandomState(SEED)
tv = tn3k[tn3k.split == "trainval"].copy()
perm = rng.permutation(len(tv))
val_ids = set(tv.iloc[perm[: int(round(0.15 * len(tv)))]].id)   # same split as Phase 1
tv["split"] = np.where(tv.id.isin(val_ids), "val", "train")
tn3k = pd.concat([tv, tn3k[tn3k.split == "test"]], ignore_index=True)

tnui = pd.concat([
    pair_images_masks(PATHS["tnui"]/s/"images", PATHS["tnui"]/s/"masks", "TNUI-2021", s)
    for s in ["train", "val", "test"]
], ignore_index=True)
tnui = add_hashes(tnui)
tnui, _ = drop_exact_duplicates(tnui, {"train": 0, "val": 1, "test": 2})

ddti = add_hashes(pair_images_masks(
    PATHS["ddti"]/"1_or_data"/"image", PATHS["ddti"]/"1_or_data"/"mask", "DDTI", "external"))
ddti, _ = drop_exact_duplicates(ddti, {"external": 0})

# episode pools: supports always disjoint from queries within an episode
POOLS = {
    "tn3k_train":  tn3k[tn3k.split == "train"].reset_index(drop=True),   # decoder training + support donors
    "tn3k_val":    tn3k[tn3k.split == "val"].reset_index(drop=True),     # early stopping
    "tn3k_test":   tn3k[tn3k.split == "test"].reset_index(drop=True),    # in-domain queries
    "tnui_train":  tnui[tnui.split == "train"].reset_index(drop=True),   # K-shot support donors (target)
    "tnui_test":   tnui[tnui.split == "test"].reset_index(drop=True),    # target queries (reported)
    "tnui_all":    tnui.reset_index(drop=True),
    "ddti_all":    ddti.reset_index(drop=True),
}
for k, v in POOLS.items():
    print(f"{k:12s} n={len(v)}")

## 2. Loading, preprocessing, and the episode sampler

Evaluation-time loading is deterministic (letterbox, binarise, normalise).
Training episodes additionally get **light joint augmentation** (hflip and
small rotation applied identically to every image and mask in the episode).

`NestedEpisodeSampler.sample()` returns K_MAX support rows and Q query rows,
disjoint; K-sub-budgets are always the first K rows of the support list, so
S1 subset S5 subset S10 subset S20 holds exactly.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

def letterbox_pair(img, msk, size):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    rm = cv2.resize(msk, (nw, nh), interpolation=cv2.INTER_NEAREST)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    cm = np.zeros((size, size), np.uint8);   cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return ci, cm

def load_item(row):
    img = read_gray(row["image"]).astype(np.float32)
    msk = (read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8)
    return letterbox_pair(img, msk, IMG_SIZE)

def to_tensor(img):  # (H,W) float in [0,1] -> normalised 3ch tensor
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

def joint_aug(imgs, msks, rng):
    do_flip = rng.rand() < 0.5
    angle = rng.uniform(-10, 10) if rng.rand() < 0.5 else 0.0
    out_i, out_m = [], []
    for im, mk in zip(imgs, msks):
        if do_flip:
            im, mk = np.ascontiguousarray(im[:, ::-1]), np.ascontiguousarray(mk[:, ::-1])
        if angle:
            M = cv2.getRotationMatrix2D((IMG_SIZE / 2, IMG_SIZE / 2), angle, 1.0)
            im = cv2.warpAffine(im, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR)
            mk = cv2.warpAffine(mk, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_NEAREST)
        if rng.rand() < 0.5:  # per-image gain
            g = rng.uniform(0.8, 1.2)
            im = np.clip(im * g, 0, 1)
        out_i.append(im.astype(np.float32)); out_m.append((mk > 0).astype(np.uint8))
    return out_i, out_m

class NestedEpisodeSampler:
    """Supports always disjoint from queries; K budgets nested by slicing."""
    def __init__(self, support_df, query_df, q=Q_QUERY, seed=SEED, same_pool_disjoint_check=True):
        self.sup = support_df.reset_index(drop=True)
        self.que = query_df.reset_index(drop=True)
        self.q = q
        self.rng = np.random.RandomState(seed)
        self.shared = len(set(self.sup.sha256) & set(self.que.sha256)) > 0
    def sample(self, kmax=K_MAX, q=None, exclude_shared=True):
        q = q or self.q
        if exclude_shared and self.shared:
            idx = self.rng.permutation(len(self.sup))
            sup_rows = self.sup.iloc[idx[:kmax]]
            taken = set(sup_rows.sha256)
            avail = self.que[~self.que.sha256.isin(taken)]
            que_rows = avail.iloc[self.rng.permutation(len(avail))[:q]]
        else:
            sup_rows = self.sup.iloc[self.rng.permutation(len(self.sup))[:kmax]]
            que_rows = self.que.iloc[self.rng.permutation(len(self.que))[:q]]
        if len(sup_rows) != kmax or len(que_rows) != q:
            raise ValueError(
                f"episode under-filled: supports {len(sup_rows)}/{kmax}, "
                f"queries {len(que_rows)}/{q} - pool too small for the protocol")
        return sup_rows.reset_index(drop=True), que_rows.reset_index(drop=True)

def episode_tensors(sup_rows, que_rows, aug=False, rng=None):
    data = [load_item(r) for _, r in pd.concat([sup_rows, que_rows]).iterrows()]
    imgs, msks = [d[0] for d in data], [d[1] for d in data]
    if aug and rng is not None:
        imgs, msks = joint_aug(imgs, msks, rng)
    ns = len(sup_rows)
    sup_x = torch.stack([to_tensor(i) for i in imgs[:ns]])
    sup_m = torch.stack([torch.from_numpy(m[None]).float() for m in msks[:ns]])
    que_x = torch.stack([to_tensor(i) for i in imgs[ns:]])
    que_m = torch.stack([torch.from_numpy(m[None]).float() for m in msks[ns:]])
    return sup_x, sup_m, que_x, que_m

# quick sampler self-check (pure pandas; safe to eyeball)
_s = NestedEpisodeSampler(POOLS["tn3k_train"], POOLS["tn3k_train"])
_a, _b = _s.sample()
assert len(set(_a.sha256) & set(_b.sha256)) == 0
_nested = NestedEpisodeSampler(POOLS["tnui_train"], POOLS["tnui_test"])
_s2, _q2 = _nested.sample()
print("sampler self-check OK | nested support order stable | shared pool:",
      _s.shared, _nested.shared)
del _s, _a, _b, _nested, _s2, _q2

## 3. Frozen Phase 1 encoder as multi-scale feature extractor

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class Phase1UNet(nn.Module):
    """Identical architecture to the Phase 1 checkpoint (state-dict compatible)."""
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e1": e1, "e2": e2, "e3": e3, "e4": e4}

backbone = Phase1UNet().to(DEVICE)
_ck = torch.load(CKPT_PATH, map_location=DEVICE)
backbone.load_state_dict(_ck["model"], strict=True)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)
print(f"loaded Phase 1 weights (which={_ck.get('which')}, val Dice={_ck.get('val_dice'):.4f}) "
      "-> encoder FROZEN, BN in eval mode")

@torch.no_grad()
def encode(x):
    with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        f = backbone.features(x)
    return {k: v.float() for k, v in f.items()}   # fp32 for stable prototypes

_x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
_f = encode(_x)
print({k: tuple(v.shape) for k, v in _f.items()})

## 4. Support-conditioned prototypes and the fusion decoder

Per scale l: masked-average foreground/background prototypes from the
(nested) support set; query score maps are negative squared Euclidean
distances. The decoder is a small 6-channel conv head at 48x48 that learns
how to blend the score maps (trained episodically on TN3K train). Setting
`use_decoder=False` bypasses it (equally-weighted fg-bg logit) - the
decoder ablation.

In [ ]:
SCALE_HW = {"e1": 96, "e2": 48, "e3": 24, "e4": 12}
N_SCORE = 2 * len(PROTO_SCALES)

def masked_proto(feats, masks):
    """feats (S,C,h,w) at scale; masks (S,1,H,W) at input res -> (C,) fg/bg prototypes."""
    h, w = feats.shape[-2:]
    m = F.interpolate(masks, size=(h, w), mode="nearest")
    fg_den = m.sum(dim=(0, 2, 3)) + 1e-6
    bg_den = (1 - m).sum(dim=(0, 2, 3)) + 1e-6
    pf = (feats * m).sum(dim=(0, 2, 3)) / fg_den
    pb = (feats * (1 - m)).sum(dim=(0, 2, 3)) / bg_den
    return pf, pb

def neg_sq_dist(feats, proto):
    d2 = (feats - proto.view(1, -1, 1, 1)).pow(2).sum(1, keepdim=True)
    return -d2

def score_maps(que_feats, pf_pb_per_scale):
    outs = []
    for name in PROTO_SCALES:
        qf = que_feats[name]
        pf, pb = pf_pb_per_scale[name]
        outs.append(F.interpolate(neg_sq_dist(qf, pf), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
        outs.append(F.interpolate(neg_sq_dist(qf, pb), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
    return torch.cat(outs, dim=1)     # (Q, N_SCORE, 48, 48)

class FusionDecoder(nn.Module):
    def __init__(self, cin=N_SCORE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1))
    def forward(self, s):
        return self.net(s)

def predict_logits(que_feats, pf_pb_per_scale, decoder=None):
    s = score_maps(que_feats, pf_pb_per_scale)
    if decoder is None:                                  # direct prototype matching
        fg = s[:, 0::2].sum(1, keepdim=True)
        bg = s[:, 1::2].sum(1, keepdim=True)
        logit = fg - bg
    else:
        logit = decoder(s)
    return F.interpolate(logit, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

def prototypes_for_K(sup_feats, sup_m, K):
    return {name: masked_proto(sup_feats[name][:K], sup_m[:K]) for name in PROTO_SCALES}

print("score channels:", N_SCORE, "| decoder params:",
      sum(p.numel() for p in FusionDecoder().parameters()))

## 5. Loss and per-case metrics


In [ ]:
def soft_dice_loss_logits(logits, targets, eps=1e-6):
    p = torch.sigmoid(logits)
    num = 2 * (p * targets).sum(dim=(1, 2, 3)) + eps
    den = p.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    return (1 - num / den).mean()

def seg_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + soft_dice_loss_logits(logits, targets)

def dice_np(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = int((pred & gt).sum()); fp = int((pred & ~gt).sum()); fn = int((~pred & gt).sum())
    if (2 * tp + fp + fn) == 0:
        return 1.0 if not (gt.any() or pred.any()) else 0.0
    return (2 * tp) / (2 * tp + fp + fn)

def iou_np(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = int((pred & gt).sum()); fp = int((pred & ~gt).sum()); fn = int((~pred & gt).sum())
    if (tp + fp + fn) == 0:
        return 1.0 if not (gt.any() or pred.any()) else 0.0
    return tp / (tp + fp + fn)

## 6. Smoke test - ONE episode end-to-end before training

Expected: prototypes from 20 supports, scores (Q,6,48,48), logits
(Q,1,384,384), finite loss. **If anything here prints NaN or a wrong shape,
stop and paste the output.**

In [ ]:
decoder = FusionDecoder().to(DEVICE)
sampler = NestedEpisodeSampler(POOLS["tn3k_train"], POOLS["tn3k_train"], seed=SEED)
sup_rows, que_rows = sampler.sample()
sup_x, sup_m, que_x, que_m = episode_tensors(sup_rows, que_rows)
sup_x, sup_m = sup_x.to(DEVICE), sup_m.to(DEVICE)
que_x, que_m = que_x.to(DEVICE), que_m.to(DEVICE)

with torch.no_grad():
    sf = encode(sup_x); qf = encode(que_x)
protos = prototypes_for_K(sf, sup_m, K_MAX)
sm = score_maps(qf, protos)
logits = predict_logits(qf, protos, decoder)
loss = seg_loss(logits, que_m)
print("supports:", tuple(sup_x.shape), "queries:", tuple(que_x.shape))
print("score maps:", tuple(sm.shape), "logits:", tuple(logits.shape))
print("loss:", float(loss), "| finite:", torch.isfinite(loss).item())

# direct-matching overlay sanity (no decoder)
with torch.no_grad():
    d_logits = predict_logits(qf, protos, decoder=None)
    d_pred = (torch.sigmoid(d_logits) >= PRED_THRESH)[0, 0].cpu().numpy()
img0 = ((que_x[0] * STD.to(DEVICE) + MEAN.to(DEVICE)).clamp(0, 1))[0].cpu().numpy()
gt0 = que_m[0, 0].cpu().numpy()
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img0, cmap="gray"); ax[0].set_title("query"); ax[0].axis("off")
ax[1].imshow(gt0, cmap="gray"); ax[1].set_title("GT"); ax[1].axis("off")
ax[2].imshow(img0, cmap="gray"); ax[2].imshow(d_pred, cmap="jet", alpha=0.35)
ax[2].set_title("direct-matching pred (untrained)"); ax[2].axis("off")
plt.tight_layout(); plt.show()

## 7. Episodic decoder training (TN3K train episodes only)

Per episode: random K sampled uniformly from the budget list (so the decoder
never overfits one budget), prototypes from nested supports, query loss
BCE+Dice. Encoder frozen throughout. Validation = mean query Dice over
100 held-out TN3K val episodes (with decoder); early stopping.

In [ ]:
def run_episodes(n_ep, support_pool, query_pool, decoder_ref, train, rng):
    sm = NestedEpisodeSampler(support_pool, query_pool, seed=int(rng.randint(1e9)))
    tot_loss, tot_dice, cnt = 0.0, [], 0
    for _ in range(n_ep):
        sup_rows, que_rows = sm.sample()
        aug = train and USE_DEC_AUG
        sup_x, sup_m, que_x, que_m = episode_tensors(sup_rows, que_rows, aug=aug, rng=rng)
        sup_x, sup_m = sup_x.to(DEVICE), sup_m.to(DEVICE)
        que_x, que_m = que_x.to(DEVICE), que_m.to(DEVICE)
        K = int(rng.choice(K_LIST))
        with torch.no_grad():
            sf = encode(sup_x); qf = encode(que_x)
        protos = prototypes_for_K(sf, sup_m, K)
        if train:
            decoder_ref.train()
            logits = predict_logits(qf, protos, decoder_ref)
            loss = seg_loss(logits, que_m)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        else:
            decoder_ref.eval()
            with torch.no_grad():
                logits = predict_logits(qf, protos, decoder_ref)
                loss = seg_loss(logits, que_m)
        pred = (torch.sigmoid(logits) >= PRED_THRESH).cpu().numpy()
        gt = que_m.cpu().numpy()
        for i in range(len(pred)):
            tot_dice.append(dice_np(pred[i, 0], gt[i, 0]))
        tot_loss += float(loss); cnt += 1
    return tot_loss / max(cnt, 1), float(np.mean(tot_dice))

opt = torch.optim.AdamW(decoder.parameters(), lr=LR_DEC, weight_decay=1e-4)
rng_tr = np.random.RandomState(SEED + 1)
best, best_ep, wait = -1.0, -1, 0
hist = []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    tr_l, tr_d = run_episodes(N_EP_TRAIN, POOLS["tn3k_train"], POOLS["tn3k_train"], decoder, True, rng_tr)
    va_l, va_d = run_episodes(N_EP_VAL, POOLS["tn3k_train"], POOLS["tn3k_val"], decoder, False, rng_tr)
    improved = va_d > best + 1e-4
    if improved:
        best, best_ep, wait = va_d, ep, 0
        torch.save({"decoder": decoder.state_dict(), "epoch": ep, "val_dice": va_d,
                    "config": {"K_LIST": K_LIST, "Q_QUERY": Q_QUERY,
                               "scales": PROTO_SCALES, "score_hw": SCORE_HW}},
                   OUTPUT_ROOT / "best_fusion_decoder.pt")
    else:
        wait += 1
    hist.append({"epoch": ep, "train_loss": tr_l, "train_dice": tr_d,
                 "val_loss": va_l, "val_dice": va_d})
    print(f"ep {ep:02d} | train loss {tr_l:.4f} dice {tr_d:.4f} | val loss {va_l:.4f} dice {va_d:.4f}"
          + ("  *" if improved else ""))
    if wait >= PATIENCE:
        print("early stopping at epoch", ep)
        break
hist_df = pd.DataFrame(hist)
hist_df.to_csv(OUTPUT_ROOT / "decoder_training_history.csv", index=False)
print(f"best val episodic Dice {best:.4f} @ ep {best_ep} | {(time.time()-t0)/60:.1f} min")

_ck2 = torch.load(OUTPUT_ROOT / "best_fusion_decoder.pt", map_location=DEVICE)
decoder.load_state_dict(_ck2["decoder"]); decoder.eval()
print("reloaded best decoder (val Dice %.4f)" % _ck2["val_dice"])

## 8. Evaluation - nested budgets, three regimes

200 episodes per condition; every episode evaluates all four budgets on the
same queries with nested supports. Conditions:

- **In-domain:** supports TN3K train -> queries TN3K test
- **Zero-shot TNUI / DDTI:** supports TN3K train -> queries TNUI test / DDTI
- **K-shot TNUI:** supports TNUI train donors -> queries TNUI test
- **K-shot DDTI:** supports and queries drawn disjointly from DDTI

`variant=decoder` (proposed) and `variant=direct` (ablation: no learned
decoder) are both reported.

In [ ]:
def eval_condition(name, support_pool, query_pool, n_ep=N_EP_EVAL, seed=SEED):
    sm = NestedEpisodeSampler(support_pool, query_pool, seed=seed)
    rows = []
    t0 = time.time()
    for ep_i in range(n_ep):
        sup_rows, que_rows = sm.sample()
        sup_x, sup_m, que_x, que_m = episode_tensors(sup_rows, que_rows)
        sup_x, sup_m = sup_x.to(DEVICE), sup_m.to(DEVICE)
        que_x = que_x.to(DEVICE)
        gt = que_m.numpy()
        with torch.no_grad():
            sf = encode(sup_x); qf = encode(que_x)
            for K in K_LIST:
                protos = prototypes_for_K(sf, sup_m, K)
                for variant, dec in [("decoder", decoder), ("direct", None)]:
                    logits = predict_logits(qf, protos, dec)
                    pred = (torch.sigmoid(logits) >= PRED_THRESH).cpu().numpy()
                    for qi in range(len(pred)):
                        rows.append({"condition": name, "episode": ep_i, "K": K,
                                     "variant": variant, "query_id": que_rows.iloc[qi]["id"],
                                     "dice": dice_np(pred[qi, 0], gt[qi, 0]),
                                     "iou": iou_np(pred[qi, 0], gt[qi, 0])})
        if (ep_i + 1) % 50 == 0:
            print(f"  {name}: {ep_i+1}/{n_ep} episodes ({(time.time()-t0)/60:.1f} min)")
    return pd.DataFrame(rows)

def boot_ci_episodes(df, metric="dice", n=2000):
    ep_mean = df.groupby("episode")[metric].mean().to_numpy()
    rng = np.random.RandomState(SEED)
    s = np.array([np.mean(rng.choice(ep_mean, len(ep_mean), replace=True)) for _ in range(n)])
    return float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

conditions = [
    ("TN3K test | in-domain",      POOLS["tn3k_train"], POOLS["tn3k_test"]),
    ("TNUI test | zero-shot",      POOLS["tn3k_train"], POOLS["tnui_test"]),
    ("TNUI test | K-shot target",  POOLS["tnui_train"], POOLS["tnui_test"]),
    ("DDTI | zero-shot",           POOLS["tn3k_train"], POOLS["ddti_all"]),
    # DDTI K-shot: support donors and queries drawn from DDTI, disjoint per episode
    ("DDTI | K-shot target",       POOLS["ddti_all"],   POOLS["ddti_all"]),
]

all_rows = []
for nm, sp, qp in conditions:
    print("evaluating:", nm)
    all_rows.append(eval_condition(nm, sp, qp))
res = pd.concat(all_rows, ignore_index=True)
res.to_csv(OUTPUT_ROOT / "phase2_per_episode_metrics.csv", index=False)

summary_rows = []
for (nm, K, variant), g in res.groupby(["condition", "K", "variant"], sort=False):
    lo, hi = boot_ci_episodes(g)
    summary_rows.append({"condition": nm, "K": K, "variant": variant,
                         "dice": g.dice.mean(), "dice_lo": lo, "dice_hi": hi,
                         "iou": g.iou.mean(), "n_episodes": g.episode.nunique(),
                         "n_cases": len(g)})
summ = pd.DataFrame(summary_rows)
summ.to_csv(OUTPUT_ROOT / "phase2_summary.csv", index=False)
piv = summ.pivot_table(index=["condition", "variant"], columns="K", values="dice").round(4)
display(piv)

### 8.1 Against the Phase 1 supervised reference

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
keys = [("TN3K test | in-domain", "TN3K test (in-domain)"),
        ("TNUI test | K-shot target", "TNUI test (ext)")]
for ax, (cond, ref_key), variant in [
        (axes[0], keys[0], "decoder"),
        (axes[1], keys[1], "decoder")]:
    g = summ[(summ.condition == cond) & (summ.variant == variant)]
    ax.errorbar(g.K, g.dice, yerr=[g.dice - g.dice_lo, g.dice_hi - g.dice],
                marker="o", capsize=4, label="Phase 2 (decoder)")
    ref = SUPERVISED_REF.get(ref_key)
    if ref:
        ax.axhline(ref, ls="--", c="grey", label=f"Phase 1 supervised (full data) {ref:.3f}")
    ax.set_xscale("log", base=2); ax.set_xticks(K_LIST); ax.set_xticklabels(K_LIST)
    ax.set_xlabel("support budget K (masks)"); ax.set_ylabel("query Dice")
    ax.set_title(cond); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("Cost-of-annotation table (decoder variant):")
for cond in ["TN3K test | in-domain", "TNUI test | zero-shot",
             "TNUI test | K-shot target", "DDTI | zero-shot", "DDTI | K-shot target"]:
    g = summ[(summ.condition == cond) & (summ.variant == "decoder")]
    if len(g):
        row = {int(r.K): round(r.dice, 4) for r in g.itertuples()}
        print(f"  {cond:30s} {row}")

### 8.2 Qualitative - TNUI test query at K=1 vs K=20

In [ ]:
sm_q = NestedEpisodeSampler(POOLS["tnui_train"], POOLS["tnui_test"], seed=SEED + 7)
sup_rows, que_rows = sm_q.sample()
sup_x, sup_m, que_x, que_m = episode_tensors(sup_rows, que_rows)
sup_x, sup_m = sup_x.to(DEVICE), sup_m.to(DEVICE); que_x = que_x.to(DEVICE)
with torch.no_grad():
    sf = encode(sup_x); qf = encode(que_x)
    preds = {}
    for K in [1, 20]:
        lg = predict_logits(qf, prototypes_for_K(sf, sup_m, K), decoder)
        preds[K] = (torch.sigmoid(lg) >= PRED_THRESH).cpu().numpy()
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for r in range(3):
    img = ((que_x[r] * STD.to(DEVICE) + MEAN.to(DEVICE)).clamp(0, 1))[0].cpu().numpy()
    gt = que_m.numpy()[r, 0]
    for c, (title, arr) in enumerate([("query", img), ("GT", gt),
                                      ("K=1 pred", preds[1][r, 0]), ("K=20 pred", preds[20][r, 0])]):
        ax = axes[r, c]
        if arr.ndim == 2 and title != "query":
            ax.imshow(img, cmap="gray"); ax.imshow(arr, cmap="jet", alpha=0.35)
        else:
            ax.imshow(arr, cmap="gray")
        d = dice_np(preds[1][r, 0], gt) if title == "K=1 pred" else (dice_np(preds[20][r, 0], gt) if title == "K=20 pred" else None)
        ax.set_title(title + (f"  Dice={d:.3f}" if d is not None else ""), fontsize=9)
        ax.axis("off")
plt.suptitle("TNUI K-shot: same query, nested supports K=1 vs K=20")
plt.tight_layout(); plt.show()

## 9. Run summary and artifacts

In [ ]:
run = {
    "phase": "2 - support-conditioned prototype segmentation",
    "seed": SEED, "device": str(DEVICE),
    "checkpoint": str(CKPT_PATH),
    "config": {"K_LIST": K_LIST, "Q_QUERY": Q_QUERY, "N_EP_EVAL": N_EP_EVAL,
                   "scales": PROTO_SCALES, "score_hw": SCORE_HW,
                   "decoder_epochs_max": EPOCHS, "pred_thresh": PRED_THRESH},
    "decoder_best_val_dice": float(best), "decoder_best_epoch": int(best_ep),
    "supervised_reference_TTA": SUPERVISED_REF,
    "summary": summ.astype(str).to_dict(orient="records"),
}
with open(OUTPUT_ROOT / "run_summary.json", "w") as f:
    json.dump(run, f, indent=2)
print("Artifacts in", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size/1024:9.1f} KB")

## 10. Decision log and hand-off

**Send back:**
1. the Section 8 pivot table (Dice by condition x variant x K);
2. the Section 8.1 figures/printed cost-of-annotation table;
3. decoder training best val Dice + epochs (Section 7);
4. the Section 6 smoke-test printing (shapes, loss finite) and overlay;
5. the K=1 vs K=20 qualitative grid (Section 8.2);
6. runtimes per condition.

**Read for the paper:** the zero-shot rows show how far source-only
prototypes carry; the K-shot rows show the annotation cost of recovery;
the decoder-vs-direct rows are the prototype-fusion ablation; and the
Phase 1 reference line shows what full-data supervision still buys.
Phase 3 will attach the lesion/context representation + prototype
classifier to this segmentation backbone (TN5000 pathology labels).